# Phase 1 — Notebook 2: TF-IDF, Category Analysis & NMF

Builds feature matrices and runs all Phase 1 analytical work on the clean
corpus from `01_preprocess.ipynb`. `make_vec`, `add_bin`, and `group_key` —
all imported from `utils.py` — are shared across Steps 1, 3, and 4, which is
the main reason those three live together.

| Step | What | Key output |
|------|------|------------|
| 1 | Build TF-IDF matrices | `features/X_*.npz`, `vocab/vocab_*.csv` |
| 2 | Quality CP2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `nmf_weights.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |


## Setup

In [1]:
import sys, json, os, httpx
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import NMF
from collections import defaultdict

from openai import OpenAI

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import (load_cfg, tokens_to_str, flat_freq, make_vec,
                   add_bin, group_key, quality_report)

CFG = load_cfg()

def OUT(subdir, fname):
    p = ROOT / "OUTPUTS" / subdir / fname
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

df  = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_filtered.parquet")
ct  = CFG["tfidf"]
print(f"Loaded {len(df):,} projects  |  {df.shape[1]} columns")

Loaded 896,277 projects  |  45 columns


---
## Parameters

All tunable values live here. Edit this cell; do not edit parameter references in downstream cells.


In [2]:
# ── ANALYSIS PARAMETERS — tune here, do not edit downstream cells ────
GROUPBY_FIELD     = CFG["analysis"]["group_by"][0]
GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
                        GROUPBY_FIELD,
                        f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose project data")
MIN_SHARED        = 3      # minimum shared NMF terms for cross-group theme detection
MIN_COVERAGE      = CFG["analysis"]["min_coverage"]   # minimum groups a theme must span
N_COMPONENTS      = 8      # NMF topics per group
TOP_N_VOCAB       = 20     # top-N vocab terms fed into NMF per group
N_REPRESENTATIVE  = CFG["llm"]["n_representative_snippets"]

MODEL_LABELING        = "gpt-5.4-mini"   # per-topic labeling loop
MODEL_CURRENT_EVENTS  = "gpt-5.4"        # current events fetch (support step, not full reasoning)
MODEL_SYNTHESIS       = "gpt-5.4"        # cross-topic synthesis call

REVIEW_GROUP      = None   # set to a specific group value to pin; None = auto-selects largest

print(f"GROUPBY_FIELD     = {GROUPBY_FIELD!r}")
print(f"GROUP_DESCRIPTION = {GROUP_DESCRIPTION!r}")


GROUPBY_FIELD     = 'posted_year_quarter'
GROUP_DESCRIPTION = 'quarterly time periods showing how teacher request language has evolved over time on DonorsChoose'


---
## Step 1 — Build TF-IDF Matrices

One vectorizer fit per n-gram range on the full corpus. Trigrams are persisted
even if unused in Phase 1 analysis — Phase 2+ inherits them without re-extraction.


In [3]:
docs  = df["tokens"].apply(tokens_to_str).tolist()
# N-grams are generated here by sklearn rather than in preprocessing.
# The vocab CSVs written below are the persisted n-gram vocabulary the
# spec calls for — just produced at feature time instead of as a
# separate preprocessing step, which avoids a redundant intermediate artifact.
specs = {"X_unigram": (1,1), "X_bigram": (2,2), "X_unigram_bigram": (1,2)}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec            = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name]     = vec
    sz, nnz        = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz/(sz[0]*sz[1]):.3f}")

# Vocab artifact: feature names + IDF weights per matrix
for name, vec in vecs.items():
    (pd.DataFrame({"feature": vec.get_feature_names_out(), "idf": vec.idf_})
       .sort_values("idf")
       .to_csv(OUT("vocab", f"vocab_{name}.csv"), index=False))

feat_dir = ROOT / "OUTPUTS/features"
feat_dir.mkdir(parents=True, exist_ok=True)
for name, X in matrices.items():
    sp.save_npz(feat_dir / f"{name}.npz", X.tocsr())

META_COLS = ["project_id", "project_category", "posted_date", "funded_date",
             "metro_type_at_time_of_posting", "grade_band", "theme_version"]
meta = df[META_COLS].reset_index(drop=True)
try:    meta.to_parquet(feat_dir / "meta.parquet", index=False)
except ImportError: meta.to_csv(feat_dir / "meta.csv", index=False)

assert matrices["X_unigram"].shape[0] == len(df)

  X_unigram           : shape=(896277, 7296)  sparsity=0.991
  X_bigram            : shape=(896277, 739735)  sparsity=1.000
  X_unigram_bigram    : shape=(896277, 747031)  sparsity=1.000
  X_trigram           : shape=(896277, 425983)  sparsity=1.000


---
## Step 2 — Quality Report: Checkpoint 2


In [4]:
qr2 = quality_report(df, label="cp2", matrices=matrices,
                     save_path=OUT("quality", "quality_cp2.json"))


=======================================================  [cp2]
  Projects : 896,277
  Tok/proj : min=5  p50=61  max=197
  Vocab    : 7,297 unique tokens
  X_unigram           : shape=[896277, 7296]  sparsity=0.991
  X_bigram            : shape=[896277, 739735]  sparsity=1.000
  X_unigram_bigram    : shape=[896277, 747031]  sparsity=1.000
  X_trigram           : shape=[896277, 425983]  sparsity=1.000
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [5]:
bins       = CFG["analysis"].get("bins", [])
min_proj   = CFG["analysis"]["min_category_projects"]
top_n      = TOP_N_VOCAB

df_work    = add_bin(df, bins) if bins else df.copy()
df_work    = df_work.reset_index(drop=True)
group_cols = [GROUPBY_FIELD, "bin"] if bins else [GROUPBY_FIELD]

# Fit once; reuse across all slices
all_docs  = df_work["tokens"].apply(tokens_to_str).tolist()
vec_cat   = make_vec(2, 0.95, (1, 2))
X_full    = vec_cat.fit_transform(all_docs)
feat      = vec_cat.get_feature_names_out()
idf_vals  = vec_cat.idf_

def cat_tfidf_slice(idx):
    """Score category slice by index against the rest of the corpus."""
    rest_idx  = df_work.index.difference(idx)
    X_cat     = X_full[idx.tolist()]
    X_rest    = X_full[rest_idx.tolist()]
    cat_prev  = (X_cat  > 0).mean(axis=0).A1
    rest_prev = (X_rest > 0).mean(axis=0).A1 if len(rest_idx) else np.zeros(len(feat))
    tf        = X_cat.mean(axis=0).A1
    return (pd.DataFrame({
                "token":         feat,
                "tf":            tf,
                "idf":           idf_vals,
                "tfidf":         tf * idf_vals,
                "prevalence":    cat_prev,
                "contrast":      cat_prev - rest_prev,
                "project_count": (X_cat > 0).sum(axis=0).A1.astype(int),
            }).nlargest(top_n, "tfidf"))

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj: continue
    kd  = group_key(keys, group_cols)
    top = cat_tfidf_slice(sub.index)
    for col, val in kd.items(): top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(f"No groups met min_proj={min_proj} threshold — lower min_category_projects in params.yaml")
cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)
print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

960 rows  |  48 groups


,posted_year_quarter,token,tf,idf,tfidf,prevalence,contrast,project_count
0,2014.0_Q3.0,donation,0.009169,3.421789,0.031376,0.256260,0.170260,3725
1,2014.0_Q3.0,technology,0.009717,2.886265,0.028046,0.327738,0.179005,4764
2,2014.0_Q3.0,ipad,0.006999,3.976284,0.027828,0.170336,0.121322,2476
3,2014.0_Q3.0,common core,0.004331,5.730249,0.024818,0.076087,0.068373,1106
4,2014.0_Q3.0,level,0.008798,2.719247,0.023924,0.315493,0.138539,4586
5,2014.0_Q3.0,able,0.011509,2.060457,0.023714,0.534535,0.191341,7770
6,2014.0_Q3.0,read,0.011291,2.095689,0.023663,0.511282,0.179891,7432
7,2014.0_Q3.0,live,0.008007,2.920210,0.023381,0.272083,0.127577,3955
8,2014.0_Q3.0,reduce lunch,0.005178,4.434582,0.022964,0.113030,0.082124,1643
9,2014.0_Q3.0,improve,0.006974,3.257278,0.022717,0.206384,0.103428,3000


In [6]:
# Top contrast tokens per group — quick cross-group comparison
(cat_tfidf_df.groupby(group_cols[0])
    .apply(lambda g: g.nlargest(5, "contrast")["token"].tolist(), include_groups=False)
    .rename("top_contrast_tokens").to_frame())

,top_contrast_tokens
posted_year_quarter,
2014.0_Q3.0,"[able, read, technology, donation, class]"
2014.0_Q4.0,"[able, technology, donation, read, class]"
2015.0_Q1.0,"[able, donation, class, grade, teach]"
2015.0_Q2.0,"[able, donation, technology, teach, grade]"
2015.0_Q3.0,"[able, donation, technology, read, class]"
2015.0_Q4.0,"[able, donation, technology, class, level]"
2016.0_Q1.0,"[able, donation, technology, project, one]"
2016.0_Q2.0,"[able, donation, free, one, other]"
2016.0_Q3.0,"[free, lunch, move, one, able]"


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per category so the dominant axis in one category
does not suppress signal in others. Topics are candidates for human review —
not stable theme definitions (that is Phase 2).

The weights DataFrame records which projects load most strongly on each topic;
Step 5 uses it to select representative snippets by NMF weight rather than
randomly.


In [7]:
cn    = CFG["nmf"]
n_rep = N_REPRESENTATIVE

def nmf_one(docs):
    """Fit NMF on one slice. Returns (topics_df, W) or (None, None)."""
    vec = make_vec(ct["min_df"], ct["max_df"], (1, 2))
    X   = vec.fit_transform(docs)
    if X.shape[1] < N_COMPONENTS: return None, None
    model = NMF(n_components=N_COMPONENTS, random_state=cn["random_seed"],
                init="nndsvd", max_iter=cn["max_iter"])
    W     = model.fit_transform(X)
    vocab = vec.get_feature_names_out()
    rows  = []
    for i, comp in enumerate(model.components_):
        idx = comp.argsort()[::-1][:cn["top_words"]]
        rows.append({"topic_id": i, "top_terms": vocab[idx].tolist(),
                     "top_weights": comp[idx].tolist()})
    return pd.DataFrame(rows), W

all_topics, all_weights = [], []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj: continue
    kd     = group_key(keys, group_cols)
    docs   = sub["tokens"].apply(tokens_to_str).tolist()
    pids   = sub["project_id"].tolist()
    topics, W = nmf_one(docs)
    if topics is None: continue
    for col, val in kd.items(): topics[col] = val
    all_topics.append(topics)
    for tid in range(W.shape[1]):
        for rank, idx in enumerate(W[:, tid].argsort()[::-1][:n_rep]):
            all_weights.append({**kd, "topic_id": tid, "project_id": pids[idx],
                                "weight": float(W[idx, tid]), "rank": rank})

if not all_topics:
    raise RuntimeError(f"No groups produced NMF topics — check N_COMPONENTS={N_COMPONENTS} vs vocab size, or lower min_category_projects")
topics_df  = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)
topics_df.to_csv(OUT("analysis", "nmf_topics.csv"),   index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)
print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")
topics_df.head(6)

/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(


384 topics across 48 groups


/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(


,topic_id,top_terms,top_weights,posted_year_quarter
0,0,"[center, math, fun, kindergarten, letter, acti...","[0.6275405341967571, 0.6079353769842701, 0.576...",2014.0_Q3.0
1,1,"[technology, ipad, computer, access, app, rese...","[0.784373297784523, 0.6047762365213178, 0.5932...",2014.0_Q3.0
2,2,"[supply, room, place, hard, keep, work, feel, ...","[0.32240717563623333, 0.3042318949225226, 0.30...",2014.0_Q3.0
3,3,"[lunch, free, reduce, free reduce, reduce lunc...","[1.044021158089152, 1.0332812241467524, 1.0025...",2014.0_Q3.0
4,4,"[book, reader, read, library, level, interest,...","[0.8080995726394706, 0.7309889092895816, 0.653...",2014.0_Q3.0
5,5,"[science, understand, hand, world, explore, ex...","[0.5113782679110154, 0.3912061306114717, 0.379...",2014.0_Q3.0


In [8]:
_eligible_groups = topics_df[group_cols[0]].unique().tolist()
if REVIEW_GROUP is not None and REVIEW_GROUP not in _eligible_groups:
    raise ValueError(f"REVIEW_GROUP={REVIEW_GROUP!r} not in topics — eligible: {_eligible_groups}")
REVIEW_GROUP = REVIEW_GROUP or topics_df[group_cols[0]].value_counts().index[0]
(topics_df[topics_df[group_cols[0]] == REVIEW_GROUP]
    [["topic_id","top_terms"]]
    .assign(top_terms=lambda x: x.top_terms.apply(lambda t: ", ".join(t[:8]))))

,topic_id,top_terms
0,0,"center, math, fun, kindergarten, letter, activ..."
1,1,"technology, ipad, computer, access, app, resea..."
2,2,"supply, room, place, hard, keep, work, feel, one"
3,3,"lunch, free, reduce, free reduce, reduce lunch..."
4,4,"book, reader, read, library, level, interest, ..."
5,5,"science, understand, hand, world, explore, exp..."
6,6,"common core, core, common, standard, magazine,..."
7,7,"special, disability, education, autism, specia..."


In [9]:
# Cross-group universal themes — terms appearing in NMF topics across many groups

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i+1:]:
        if ci == cj: continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

# Sort by n_groups desc, then drop any theme whose terms are a subset of a prior theme
rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE: continue
    if not any(terms <= prior for prior in seen):
        deduped.append({"theme": ", ".join(sorted(terms)[:5]),
                        "n_groups": len(cats),
                        "categories": sorted(cats)})
        seen.append(terms)

pd.DataFrame(deduped).reset_index(drop=True)

,theme,n_groups,categories
0,"book, interest, library, love, read",46,"[2014.0_Q3.0, 2014.0_Q4.0, 2015.0_Q1.0, 2015.0..."
1,"book, interest, level, library, love",39,"[2014.0_Q3.0, 2014.0_Q4.0, 2015.0_Q1.0, 2015.0..."
2,"book, grade, interest, level, library",30,"[2014.0_Q4.0, 2015.0_Q1.0, 2015.0_Q3.0, 2016.0..."
3,"chair, comfortable, desk, flexible, flexible seat",29,"[2017.0_Q2.0, 2017.0_Q3.0, 2017.0_Q4.0, 2018.0..."
4,"income, low, low income",28,"[2014.0_Q4.0, 2016.0_Q2.0, 2016.0_Q3.0, 2017.0..."
...,...,...,...
172,"book, enjoy, grade, interest, level",6,"[2022.0_Q3.0, 2022.0_Q4.0, 2023.0_Q2.0, 2023.0..."
173,"hand, motor, skill",6,"[2022.0_Q3.0, 2022.0_Q4.0, 2023.0_Q2.0, 2023.0..."
174,"fine, motor, play, skill",6,"[2022.0_Q3.0, 2022.0_Q4.0, 2023.0_Q2.0, 2023.0..."
175,"explore, hand, problem, problem solve, skill",6,"[2023.0_Q4.0, 2024.0_Q1.0, 2024.0_Q3.0, 2025.0..."


---
## Step 5 — LLM Topic Labeling

One API call per topic on compressed input only — never raw essay text at scale.
Representative snippets are chosen by NMF weight (not random).

Prompt field limits (`top_unigrams`, `top_bigrams`, `top_nmf_terms`) are applied
as item counts. Parse failures are stored with enough metadata to debug later.

**Requires:** `pip install anthropic` + `ANTHROPIC_API_KEY` env var.
Without the package, stubs are written so downstream steps stay testable.


In [10]:
# Initiate OpenAI credentials

import os
import httpx
from openai import OpenAI

if "client" not in globals():
    from openai import OpenAI
    import httpx
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Missing OPENAI_API_KEY environment variable.")
    http_client = httpx.Client(verify=False)
    client = OpenAI(api_key=api_key, http_client=http_client)
    print("✅ OpenAI client initialized.")
else:
    print("🔁 Client already loaded; skipping setup.")

✅ OpenAI client initialized.


In [11]:
SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a JSON object, no preamble, no markdown fences."
)
PROMPT = (
    f"Group ({GROUPBY_FIELD}): {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"
    f"Return a JSON object with exactly these keys:\n"
    f"  {GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes\n"
    'coherence_flag must be one of: "coherent", "mixed", "redundant"'
)

lc = CFG["llm"]

has_essay = "essay" in df.columns
pid_text  = (df.set_index("project_id")["essay"].str[:300] if has_essay
             else df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40])))

def build_input(t_row):
    terms    = t_row["top_terms"]
    key_cols = [GROUPBY_FIELD] + (["bin"] if "bin" in t_row.index else [])
    mask     = weights_df["topic_id"] == t_row["topic_id"]
    for col in key_cols: mask &= weights_df[col] == t_row[col]
    rep_pids = (weights_df[mask].sort_values("weight", ascending=False)
                    ["project_id"].tolist()[:N_REPRESENTATIVE])
    return {
        "group_value":       t_row[GROUPBY_FIELD],
        "topic_id":  int(t_row["topic_id"]),
        "bin_line":  f"\nBin: {t_row['bin']}"
                     if "bin" in t_row.index and pd.notna(t_row.get("bin")) else "",
        "unigrams":  ", ".join([x for x in terms if " " not in x][:6]),
        "bigrams":   ", ".join([x for x in terms if " " in  x][:4]),
        "nmf_terms": ", ".join(terms[:10]),
        "snippets":  "\n".join(f"- {pid_text.get(p, '')}" for p in rep_pids),
    }

results = []
for _, t in topics_df.iterrows():
    inp  = build_input(t)
    resp = client.chat.completions.create(
        model=MODEL_LABELING,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": PROMPT.format(**inp)},
        ],
    )
    text = resp.choices[0].message.content.strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        obj = {"raw": text, "parse_error": True, "model": MODEL_LABELING,
               "model": MODEL_LABELING,
               "timestamp": datetime.now().isoformat(),
               GROUPBY_FIELD: inp["group_value"], "topic_id": inp["topic_id"]}
    results.append(obj)
    print(f"  {inp['group_value']} / topic {inp['topic_id']} "
          f"→ {obj.get('proposed_label', '?')}")

with open(OUT("analysis", "llm_topic_labels.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\n{len(results)} labels saved")

  2014.0_Q3.0 / topic 0 → Hands-on literacy and math centers for early learners
  2014.0_Q3.0 / topic 1 → Classroom technology access and digital learning tools
  2014.0_Q3.0 / topic 2 → Classroom space and supply constraints
  2014.0_Q3.0 / topic 3 → Free/Reduced Lunch Eligibility and Low-Income Student Needs
  2014.0_Q3.0 / topic 4 → Reading Access and Student Book Interest
  2014.0_Q3.0 / topic 5 → Science inquiry, problem-solving, and hands-on STEM learning
  2014.0_Q3.0 / topic 6 → Common Core nonfiction and current-events magazines
  2014.0_Q3.0 / topic 7 → Special education and disability support
  2014.0_Q4.0 / topic 0 → Low-income classroom supplies and basic math materials
  2014.0_Q4.0 / topic 1 → Reading materials and comprehension support
  2014.0_Q4.0 / topic 2 → Technology access and classroom devices
  2014.0_Q4.0 / topic 3 → Free/reduced lunch eligibility and low-income student context
  2014.0_Q4.0 / topic 4 → Special education seating and sensory support
  2014.0_Q4.

In [12]:
labels_df = pd.DataFrame(results)
labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD,"topic_id","proposed_label","coherence_flag","description"]]

,posted_year_quarter,topic_id,proposed_label,coherence_flag,description
0,2014.0_Q3.0,0,Hands-on literacy and math centers for early l...,coherent,Projects in this cluster describe classroom ce...
1,2014.0_Q3.0,1,Classroom technology access and digital learni...,coherent,Projects focused on providing students with ac...
2,2014.0_Q3.0,2,Classroom space and supply constraints,coherent,"Projects center on limited classroom space, se..."
3,2014.0_Q3.0,3,Free/Reduced Lunch Eligibility and Low-Income ...,coherent,This topic centers on students who qualify for...
4,2014.0_Q3.0,4,Reading Access and Student Book Interest,coherent,Projects centered on helping students find boo...
...,...,...,...,...,...
379,nan_Qnan,3,Hands-on creative problem solving and STEM exp...,coherent,"Projects focused on tactile, creative, hands-o..."
380,nan_Qnan,4,"Comfort, focus, and behavior support",coherent,Projects in this cluster emphasize creating a ...
381,nan_Qnan,5,Early literacy and reading practice,coherent,This topic centers on foundational literacy su...
382,nan_Qnan,6,Social-emotional regulation and sensory support,coherent,Projects in this cluster focus on helping stud...


## Current events context for synthesis — fetches once, caches for 7 days

In [13]:
from datetime import timedelta

ce_dir   = OUT("analysis", "placeholder").parent
ce_files = sorted(ce_dir.glob("current_events_*.txt"))
current_events = None

if ce_files:
    newest = ce_files[-1]
    ts_str = newest.stem.replace("current_events_", "")
    try:
        ts = datetime.strptime(ts_str, "%Y%m%d_%H%M%S")
        if datetime.now() - ts < timedelta(days=7):
            current_events = newest.read_text()
            print(f"Using cached current events from {ts.strftime('%Y-%m-%d %H:%M')} → {newest.name}")
    except ValueError:
        pass

if current_events is None:
    print("Fetching current events via web search...")
    ce_resp = client.chat.completions.create(
        model=MODEL_CURRENT_EVENTS,
        messages=[
            {"role": "system", "content": (
                "You are a policy researcher with knowledge of US K-12 education developments "
                "through early 2026. Be specific and factual. Cite named legislation, programs, "
                "or events where possible."
            )},
            {"role": "user", "content": (
                "Summarize recent developments (2025-2026) in these three areas as they relate "
                "to K-12 public schools in the United States: "
                "(1) Food insecurity and economic hardship affecting families with school-age children — "
                "including federal food assistance programs, SNAP, free and reduced lunch policy, "
                "and any broader economic pressures (inflation, unemployment, cost of living) "
                "that may affect whether students come to school fed and ready to learn. "
                "(2) AI tools, devices, and platforms in K-12 classrooms — new products, "
                "district policies, and debates about usage. "
                "(3) Student mental health funding and services — federal or state funding "
                "changes, school counselor ratios, and crisis intervention programs. "
                "Write 3-5 sentences per topic. Be specific, not general."
            )},
        ],
    )
    raw = ce_resp.choices[0].message.content
    current_events = (
        raw if isinstance(raw, str)
        else "\n\n".join(
            (b.get("text", "") if isinstance(b, dict) else getattr(b, "text", ""))
            for b in raw
            if (isinstance(b, dict) and b.get("type") == "text")
            or (hasattr(b, "type") and b.type == "text")
        )
    )
    ts_str  = datetime.now().strftime("%Y%m%d_%H%M%S")
    ce_path = ce_dir / f"current_events_{ts_str}.txt"
    ce_path.write_text(current_events)
    print(f"Saved → {ce_path.name}")

print("\n── Current events context ──────────────────────────────")
print(current_events)


Using cached current events from 2026-03-20 11:33 → current_events_20260320_113311.txt

── Current events context ──────────────────────────────
1) **Food insecurity and economic hardship**
In 2025, food insecurity risk for school-age children remained tied to household finances and federal nutrition policy, especially **SNAP**, **WIC**, and school meal access through the **National School Lunch Program (NSLP)** and **School Breakfast Program (SBP)**. The biggest practical policy issue was not a new school-meal law, but the effect of federal budget fights, state-level **Universal Free School Meals** expansions, and continued use of **Community Eligibility Provision (CEP)** in high-poverty schools to avoid application barriers and reduce stigma. Families also faced persistent pressure from higher housing, food, and childcare costs, so even where headline inflation cooled, many districts reported that students were still arriving hungry because wages and benefits had not kept pace with t

In [14]:
# Format topic labels for the prompt — exclude Awaiting Classification
topic_lines = "\n".join(
    f"  {row[GROUPBY_FIELD]} | topic {row.topic_id} | {row.proposed_label}"
    for _, row in labels_df.iterrows()
    if str(row[GROUPBY_FIELD]) != "Awaiting Classification"
)

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, insight-driven briefings for internal strategy audiences. "
    "You do not pad your answers or state the obvious."
)

SYNTHESIS_PROMPT = f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose,
grouped by {GROUPBY_FIELD} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language.

{topic_lines}

Your task: identify what is GENUINELY SURPRISING OR DISTINCTIVE in this topic landscape.

Rules:
- Every finding must be grounded in specific topic labels from the list above.
  Do not introduce concepts, themes, or absences that cannot be traced to an actual topic label.
  If you cannot name the specific group and topic label, do not make the finding.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language — it is background noise across all groups.

Focus on:
1. UNEXPECTED CONCENTRATIONS — a topic that appears in a group where it doesn't obviously
   belong. What does it reveal about how teachers are framing requests in that group?
   The test: would a reasonable person be surprised to find this topic in this group?
2. CROSS-GROUP SIGNATURES — a specific program, vendor, pedagogy, or need that shows up
   as a recognizable topic in multiple groups. Name every group it appears in and explain
   why the pattern is meaningful. Look carefully for named programs or institutions.
3. NOTABLE ABSENCES — a topic you would expect given the other topics already present in the
   SAME group, but that is missing. The bar is high: you must be able to point to at least
   two specific topics within that same group that make the gap conspicuous. Do not cite
   topics from other groups as evidence. Do not flag an absence simply because a theme
   exists elsewhere — only flag it if the gap is visible from within the group itself.

Where topics touch on AI tools in the classroom, student hunger or food insecurity, or student
mental health and regulation needs, note this explicitly — but only where the topic genuinely
warrants it. Use the current events context below to add specificity where relevant (for example,
if a topic clusters around food needs, note recent SNAP policy changes; if a topic clusters around
classroom technology, note recent AI-in-schools developments). Do not use current events to 
generate findings, infer unmet need, or introduce themes not already grounded in the discovered 
topic labels.

Format as three labeled sections. Under each, write 2-5 findings as short paragraphs.
Be specific — name the group and topic label every time. Do not use bullet points.

CURRENT EVENTS CONTEXT:
{current_events}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": SYNTHESIS_SYSTEM},
        {"role": "user",   "content": SYNTHESIS_PROMPT},
    ],
)
synthesis = resp.choices[0].message.content.strip()

with open(OUT("analysis", "llm_synthesis.txt"), "w") as f:
    f.write(synthesis)

print(synthesis)

### 1. UNEXPECTED CONCENTRATIONS

In **2015.0_Q1.0 | topic 1 | Technology-integrated professional development**, the distinctive feature is not technology alone but that teacher essays appear to cluster around **professional development** as a fundable classroom need. In this landscape, technology topics are usually framed as student-facing access or devices—compare **2014.0_Q3.0 | topic 1 | Classroom technology access and digital learning tools** and **2015.0_Q2.0 | topic 1 | Technology Access and Digital Learning Tools**. Seeing PD itself become a topic is unusual because it suggests teachers were not just asking for hardware; they were explicitly framing implementation capacity as part of the request.

In **2014.0_Q4.0 | topic 7 | School music performance and instrumental ensemble programs**, the concentration is surprising because the surrounding 2014 topics are dominated by core-academic and need-based framing—reading, science, kindergarten centers, special education seating. A cl